# SVDKL for tomato metabolite prediction

SVDKL predicts tomato metabolite levels from genomic structural variant genotypes using deep kernel learning. It combines a relationship kernel across deletion variants with a Transformer-weighted kernel built from 500 variants selected within training. This notebook provides an interactive workflow for 36 metabolite traits: choose a pretrained model, load the example data, and generate predictions. The example contains 324 tomato accessions, including 65 test accessions used for comparison plots and evaluation. Results appear as tables and figures with download buttons. Three model sizes are available, starting with SVDKL-0.01M. Additional sections support training, evaluation, and redrawing saved predictions. Prediction runs on CPU, while training can use a GPU.

[GitHub: source code and usage](https://github.com/wyqmath/SVDKL) · [Zenodo: pretrained model weights](https://zenodo.org/records/23669233)


In [ ]:
# @title ## 1. Set up SVDKL { display-mode: "form" }
# @markdown Load the SVDKL code and install the dependencies used by the following steps. The implementation and local running instructions are available on [GitHub](https://github.com/wyqmath/SVDKL). Prediction supports CPU execution; choose a GPU runtime for training. When setup finishes, the notebook displays the active package versions and available device.
from pathlib import Path
import importlib.metadata
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/wyqmath/SVDKL.git"
try:
    in_colab = importlib.util.find_spec("google.colab") is not None
except (ModuleNotFoundError, ValueError):
    in_colab = False
required = ["src/model.py", "src/train.py", "src/predict.py", "src/score.py", "src/prepare_data.py"]
candidates = [Path.cwd(), Path.cwd().parent]
REPO_DIR = next((path for path in candidates if (path / "src/model.py").is_file()),
                Path("/content/SVDKL") if in_colab else Path.cwd() / "SVDKL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
missing = [name for name in required if not (REPO_DIR / name).is_file()]
if missing:
    raise RuntimeError("The repository is incomplete: " + ", ".join(missing))
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
packages = ("numpy", "scipy", "torch", "threadpoolctl", "matplotlib", "ipywidgets")
missing_packages = [name for name in packages if importlib.util.find_spec(name) is None]
if missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages], check=True)
import numpy as np
import torch
from src.train import get_benchmark
from src import model as model_module

if not hasattr(model_module, "load_parameter_checkpoint"):
    raise RuntimeError("Update the repository files and restart the runtime before loading these weights.")

plan = get_benchmark()
print("SVDKL runtime ready.")
for package in packages:
    print(package, importlib.metadata.version(package))
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import csv
import json
import pickle
import re
import shutil
import tempfile
import urllib.parse
import urllib.request

ZENODO_RECORDS = {'SVDKL-0.01M': '23669233', 'SVDKL-4.35M': '23669233', 'SVDKL-46.46M': '23669233'}
TRAIT_NAMES = {
    "SlFM0030": "Cysteine",
    "SlFM0047": "L-Isoleucine",
    "SlFM0048": "Ornithine",
    "SlFM0069": "Salicylic acid",
    "SlFM0216": "Methoxy indoleacetic acid",
    "SlFM0222": "Chalcone",
    "SlFM0244": "Acetyl-serotonin",
    "SlFM0259": "2'-Deoxycytidine",
    "SlFM0275": "L-Homocarnosine",
    "SlFM0336": "ABA",
    "SlFM0346": "Homocystine",
    "SlFM0386": "N-(9H-Purin-6-ylcarbamoyl)threonine",
    "SlFM0416": "IAA-Asp",
    "SlFM0421": "Catechin",
    "SlFM0430": "N-Sinapoylputrescine",
    "SlFM0438": "2'-O-methylguanosine",
    "SlFM0502": "N-Feruloyltyramine",
    "SlFM0533": "p-Coumaric acid 4-O-hexoside",
    "SlFM0555": "GA7-1",
    "SlFM0577": "4-Coumaroylquinic acid",
    "SlFM0625": "Adenosine 3'-monophosphate",
    "SlFM0716": "Sinapaldehyde glucoside",
    "SlFM0778": "S-Adenosyl-L-homocysteine",
    "SlFM0982": "Naringenin C-hexoside",
    "SlFM1384": "Quercetin O-malonylhexoside",
    "SlFM1470": "Cyanidin 3-rutinoside",
    "SlFM1542": "Petunidin 3-O-rutinoside",
    "SlFM1553": "N', N'', N'''-Di-p-coumaroyl, sinapoylagmatine",
    "SlFM1619": "N', N'', N'''-Diferuloyl, caffeoylspermidine",
    "SlFM1621": "N', N'', N'''-p-hydroxybenxoyl,caffeoyl, feruloylagmatine",
    "SlFM1804": "Dihydroxy tomatidine-O-hexosyl-O-rhamnoside",
    "SlFM1826": "Delphinidin 3-sophoroside-5-rhamnoside",
    "SlFM1952": "Ferulylpelargonidin di-O-hexosyl-O-pentoside",
    "SlFM1980": "Habrochaitoside A",
    "SlFM1984": "Lycoperoside H",
    "SlFM1988": "Lycoperoside C"
}


def download_to(url, target):
    from IPython.display import display
    import ipywidgets as widgets

    parsed = urllib.parse.urlsplit(url.strip())
    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        raise ValueError("Use a complete download link starting with http:// or https://.")
    target = Path(target)
    progress = widgets.FloatProgress(min=0, max=1, value=0, description="Downloading:")
    status = widgets.HTML(value="Connecting…")
    display(widgets.VBox([progress, status]))
    try:
        with urllib.request.urlopen(url.strip(), timeout=60) as response, target.open("xb") as output:
            length = response.headers.get("Content-Length")
            total = int(length) if length and length.isdigit() else None
            received = 0
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                output.write(chunk)
                received += len(chunk)
                progress.value = min(received / total, 1) if total else 0
                status.value = (f"{received / 1048576:.1f} / {total / 1048576:.1f} MB"
                                if total else f"{received / 1048576:.1f} MB received")
            if total is not None and received != total:
                raise OSError("Download was interrupted. Run Load model again.")
            supplied_name = response.headers.get_filename()
        progress.value, progress.bar_style = 1, "success"
        status.value = "Download complete. Loading model…" if target.suffix == ".pt" else "Download complete."
        return supplied_name
    except Exception:
        progress.bar_style = "danger"
        status.value = "Download failed. Check the connection and run this step again."
        raise


def read_zenodo_json(url):
    parsed = urllib.parse.urlsplit(url)
    if parsed.scheme != "https" or parsed.hostname != "zenodo.org" or parsed.username or parsed.password:
        raise ValueError("Use the model download link from Zenodo.")
    request = urllib.request.Request(url, headers={"Accept": "application/json", "User-Agent": "SVDKL"})
    with urllib.request.urlopen(request, timeout=60) as response:
        return json.load(response)


def zenodo_model_files(record_id):
    record_id = str(record_id)
    if not re.fullmatch(r"[1-9][0-9]*", record_id):
        raise ValueError("The model download link is not configured. Choose another model or load a model from a file.")
    record_url = f"https://zenodo.org/api/records/{record_id}"
    record = read_zenodo_json(record_url)
    if str(record.get("id")) != record_id:
        raise ValueError("The download does not match the selected model. Choose the model again or load a model from a file.")
    files = record.get("files")
    legacy = isinstance(files, list)
    if not legacy:
        files = files.get("entries") if isinstance(files, dict) else None
        if files is None:
            files = read_zenodo_json(record_url + "/files").get("entries")
    if isinstance(files, dict):
        files = [dict(value, key=key) for key, value in files.items()]
    if not isinstance(files, list) or not files:
        raise RuntimeError("No model download is available. Load local model or try again later.")
    return files, legacy


def zenodo_file(files, legacy, filename):
    matches = [item for item in files if item.get("key") == filename]
    if len(matches) != 1:
        raise ValueError(f"The model release must contain exactly one {filename} file.")
    item = matches[0]
    if item.get("status") not in (None, "completed"):
        raise RuntimeError(f"The Zenodo file for {filename} is not ready for download.")
    links = item.get("links", {})
    url = links.get("content")
    if not url and legacy:
        url = links.get("self")
    elif not url and links.get("self"):
        details_url = urllib.parse.urljoin("https://zenodo.org/", links["self"])
        url = read_zenodo_json(details_url).get("links", {}).get("content")
    if not url:
        raise RuntimeError(f"No public download is available for {filename}.")
    url = urllib.parse.urljoin("https://zenodo.org/", url)
    parsed = urllib.parse.urlsplit(url)
    if parsed.scheme != "https" or parsed.hostname != "zenodo.org" or parsed.username or parsed.password:
        raise ValueError("Use the model download link from Zenodo.")
    return url, item.get("size")


def zenodo_trait_file(files, legacy, trait):
    return zenodo_file(files, legacy, f"{trait}.pt")


def check_weight_file(path, model, trait):
    from src.train import validate_plan

    path = Path(path)
    message = f"The model could not be loaded. Select {model} and run Load model again. Use Load local model to select a matching file."
    if not path.is_file():
        raise FileNotFoundError(f"Load the weights for {model}, or train this trait before running prediction.")
    if path.name != f"{trait}.pt":
        raise ValueError(message)
    if path.parent.name.startswith("SVDKL-") and path.parent.name != model:
        raise ValueError(message)
    try:
        model_config, network_config = validate_plan(plan, model)
        state = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
        model_module._check_inference_parameters(state, model_config, network_config)
    except (ValueError, RuntimeError, TypeError, KeyError, OSError, EOFError, pickle.UnpicklingError):
        raise ValueError(message) from None


def prepare_model(model, location):
    from src.train import validate_plan

    validate_plan(plan, model)
    if not location.strip() and any((Path("weights") / model / f"{trait}.pt").is_file() for trait in plan["traits"]):
        location = str(Path("weights") / model)
    if location.strip():
        location = Path(location.strip()).expanduser()
        if location.is_file():
            trait = location.stem
            if trait not in plan["traits"]:
                raise ValueError("Choose a metabolite trait from the supported trait list, then load its model file.")
            paths = {trait: location}
            directory = location.parent
        elif location.is_dir():
            directories = [directory for directory in (location / model, location)
                           if any((directory / f"{trait}.pt").is_file() for trait in plan["traits"])]
            if len(directories) != 1:
                raise ValueError(f"The model could not be loaded. Select {model} and run Load model again. Use Load local model to select a matching file.")
            directory = directories[0]
            paths = {trait: directory / f"{trait}.pt" for trait in plan["traits"]
                     if (directory / f"{trait}.pt").is_file()}
        else:
            raise FileNotFoundError(f"Load the weights for {model}, or train this trait before running prediction.")
        first_trait = next(iter(paths))
        check_weight_file(paths[first_trait], model, first_trait)
        model_root = directory.parent if directory.name == model else directory
        return {"model": model, "source": "local", "paths": paths, "root": model_root}
    record_id = ZENODO_RECORDS.get(model)
    if not record_id:
        return {"model": model, "source": "unavailable", "paths": {}, "root": None}
    try:
        files, legacy = zenodo_model_files(record_id)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f"Zenodo returned HTTP {error.code}. Check the model download link in the README, then retry Load model or use Load local model.") from None
    except (urllib.error.URLError, TimeoutError) as error:
        raise RuntimeError(f"The selected model could not be downloaded. Run Load model again, or load the weights for {model} from a file.") from None
    return {"model": model, "source": "zenodo", "files": files, "legacy": legacy,
            "root": Path("weights") / f"zenodo_{record_id}",
            "archive": any(item.get("key") == f"{model}.zip" for item in files)}



def load_model_archive(selection):
    import zipfile

    model = selection["model"]
    destination = selection["root"] / model
    expected = {f"weights/{model}/{trait}.pt": trait for trait in plan["traits"]}
    if all((destination / f"{trait}.pt").is_file() for trait in plan["traits"]):
        return
    url, size = zenodo_file(selection["files"], selection["legacy"], f"{model}.zip")
    selection["root"].mkdir(parents=True, exist_ok=True)
    try:
        with tempfile.TemporaryDirectory(dir=selection["root"]) as temporary:
            downloaded = Path(temporary) / f"{model}.zip"
            name = download_to(url, downloaded)
            if name is not None and name != f"{model}.zip":
                raise ValueError("The downloaded file does not match the selected model.")
            if size is not None and downloaded.stat().st_size != int(size):
                raise ValueError("The model download is incomplete.")
            staged = Path(temporary) / "model"
            staged.mkdir()
            print(f"Unpacking {model}…")
            with zipfile.ZipFile(downloaded) as archive:
                members = archive.infolist()
                if len(members) != len(expected) or {item.filename for item in members} != set(expected):
                    raise ValueError("The model files do not match the 36 supported metabolite traits.")
                for member in members:
                    if member.is_dir() or member.flag_bits & 1 or member.file_size <= 0:
                        raise ValueError("The model contains an unsupported file.")
                    target = staged / f"{expected[member.filename]}.pt"
                    with archive.open(member) as source, target.open("xb") as output:
                        shutil.copyfileobj(source, output, length=1024 * 1024)
            check_weight_file(staged / f"{plan['traits'][0]}.pt", model, plan["traits"][0])
            destination.mkdir(exist_ok=True)
            for trait in plan["traits"]:
                target = destination / f"{trait}.pt"
                if not target.exists():
                    os.link(staged / f"{trait}.pt", target)
    except (ValueError, OSError, RuntimeError, zipfile.BadZipFile):
        raise RuntimeError(f"The model could not be loaded. Select {model} and run Load model again, or use Load local model.") from None


def model_for_trait(selection, trait):
    if trait not in plan["traits"]:
        raise ValueError("Choose a metabolite trait from the supported trait list.")
    model = selection["model"]
    if selection["source"] in {"local", "trained"}:
        path = selection["paths"].get(trait)
        if path is None or not path.is_file():
            raise FileNotFoundError(f"Load the weights for {model}, or train this trait before running prediction.")
        if selection["source"] == "local":
            check_weight_file(path, model, trait)
        return path
    if selection["source"] == "unavailable":
        raise RuntimeError(f"Load the weights for {model}, then run prediction again.")
    if selection.get("archive"):
        load_model_archive(selection)
        target = selection["root"] / model / f"{trait}.pt"
        check_weight_file(target, model, trait)
        return target
    try:
        url, size = zenodo_trait_file(selection["files"], selection["legacy"], trait)
    except (ValueError, RuntimeError, urllib.error.URLError, TimeoutError):
        raise RuntimeError(f"Load the weights for {model}, or train this trait before running prediction.") from None
    target = selection["root"] / model / f"{trait}.pt"
    if target.exists():
        if size is not None and target.stat().st_size != int(size):
            raise ValueError(f"The model could not be loaded. Select {model} and run Load model again. Use Load local model to select a matching file.")
        check_weight_file(target, model, trait)
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=target.parent) as temporary:
        downloaded = Path(temporary) / f"{trait}.pt"
        try:
            supplied_name = download_to(url, downloaded)
        except Exception as error:
            raise RuntimeError(f"The model could not be downloaded. Run Load model again, or load the weights for {model} from a file.") from None
        if supplied_name is not None and supplied_name != f"{trait}.pt":
            raise ValueError(f"The model could not be loaded. Select {model} and run Load model again. Use Load local model to select a matching file.")
        if size is not None and downloaded.stat().st_size != int(size):
            raise ValueError(f"The model could not be loaded. Select {model} and run Load model again. Use Load local model to select a matching file.")
        check_weight_file(downloaded, model, trait)
        os.link(downloaded, target)
    return target


def unique_output(path):
    path = Path(path)
    candidate, number = path, 2
    while candidate.exists():
        candidate = path.with_name(f"{path.stem}_run{number}{path.suffix}")
        number += 1
    return candidate


def save_download(path):
    import base64
    from html import escape
    from IPython.display import HTML, FileLink, display

    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError("Run the step that generates this file again, then select Download.")
    if in_colab:
        from google.colab import files
        files.download(str(path))
    elif path.stat().st_size > 32 * 1024 * 1024:
        display(FileLink(str(path)))
    else:
        mime = "image/png" if path.suffix == ".png" else "text/csv" if path.suffix == ".csv" else "application/octet-stream"
        encoded = base64.b64encode(path.read_bytes()).decode("ascii")
        display(HTML(f'<a download="{escape(path.name, quote=True)}" href="data:{mime};base64,{encoded}">Save {escape(path.name)}</a>'))


def file_download_buttons(items, title):
    from IPython.display import display
    import ipywidgets as widgets

    output = widgets.Output()

    def download(path):
        with output:
            output.clear_output(wait=True)
            save_download(path)

    buttons = {}
    for label, path in items:
        path = Path(path)
        if not path.is_file():
            raise FileNotFoundError("Run prediction or evaluation again before downloading the results.")
        button = widgets.Button(description=label, tooltip=title, icon="download",
                                layout=widgets.Layout(width="auto"))
        button.on_click(lambda _, saved=path: download(saved))
        buttons[label] = button
    display(widgets.HBox(list(buttons.values())), output)
    return buttons


def load_example_input(location="data/dataset_inputs.npz"):
    global prepared, data_selection, prediction_result, input_revision

    prepared, data_selection, prediction_result = None, None, None
    location = Path(location).expanduser()
    if not location.is_file():
        raise FileNotFoundError("Click Prepare example data in Input data, or choose Load local data, then run prediction again.")
    expected = {"dosage", "accessions", "traits", "phenotypes", "marker_id", "chrom", "pos", "svlen"}
    with np.load(location, allow_pickle=False) as data:
        if set(data.files) != expected:
            raise ValueError("Load input data created by the optional data preparation step.")
        np.testing.assert_array_equal(data["accessions"], plan["accessions"])
        np.testing.assert_array_equal(data["traits"], plan["traits"])
        if data["dosage"].shape != (324, 106362) or data["phenotypes"].shape != (324, 36):
            raise ValueError("Load input data for the 324 accessions and 36 supported metabolite traits.")
    prepared = location
    input_revision = globals().get("input_revision", 0) + 1
    data_selection = {"path": prepared, "revision": input_revision}
    print(f"Input data loaded: {len(plan['accessions'])} accessions and {len(plan['traits'])} metabolite traits.")
    from html import escape
    from IPython.display import HTML, display
    choices = "".join(f"<li>{escape(trait)} — {escape(TRAIT_NAMES[trait])}</li>" for trait in plan["traits"])
    display(HTML("<details><summary>Available metabolite traits</summary><ul>" + choices + "</ul></details>"))
    return prepared



def prepare_example_input():
    data_dir = Path("data")
    data_dir.mkdir(exist_ok=True)
    output = data_dir / "dataset_inputs.npz"
    if not output.is_file():
        sources = (("706.sv.vcf.gz", "http://solomics.agis.org.cn/tomato/ftp/genotypes/706.sv.vcf.gz"),
                   ("pheno_exp_meta.txt", "http://solomics.agis.org.cn/tomato/ftp/expression/pheno_exp_meta.txt"))
        for name, url in sources:
            target = data_dir / name
            if not target.is_file():
                with tempfile.TemporaryDirectory(dir=data_dir) as temporary:
                    downloaded = Path(temporary) / name
                    try:
                        download_to(url, downloaded)
                    except Exception:
                        raise RuntimeError(f"Could not download {name}. Use its original publication link in the README, place it in data/, and run Prepare example data again.") from None
                    if not downloaded.stat().st_size:
                        raise ValueError(f"Downloaded source file is empty: {name}. Download it again from its original provider.")
                    os.link(downloaded, target)
        subprocess.run([sys.executable, "-m", "src.prepare_data", "--vcf", str(data_dir / "706.sv.vcf.gz"),
                        "--phenotypes", str(data_dir / "pheno_exp_meta.txt"), "--output", str(output)],
                       check=True, stdout=subprocess.PIPE)
    load_example_input(output)
    print("Example input ready. Continue with Run prediction.")
    return output


def show_example_data_action():
    import ipywidgets as widgets
    from IPython.display import display

    button = widgets.Button(description="Prepare example data", layout=widgets.Layout(width="auto"))
    output = widgets.Output()

    def prepare(_):
        button.disabled = True
        with output:
            output.clear_output(wait=True)
            try:
                prepare_example_input()
            except (ValueError, RuntimeError, OSError, subprocess.CalledProcessError) as error:
                print(f"{error} Check the source files and run Prepare example data again.")
            finally:
                button.disabled = False

    button.on_click(prepare)
    display(button, output)
    return button


def show_advanced_model(model):
    import ipywidgets as widgets
    from IPython.display import display

    location = widgets.Text(description="Model file:", placeholder="A model file or folder already in this session", layout=widgets.Layout(width="95%"))
    button = widgets.Button(description="Load local model", layout=widgets.Layout(width="auto"))
    output = widgets.Output()

    def load(_):
        global model_selection, prediction_result
        with output:
            output.clear_output(wait=True)
            model_selection, prediction_result = None, None
            try:
                if model != model_size:
                    raise RuntimeError("Rerun step 2 after changing model size.")
                if not location.value.strip():
                    raise ValueError("Enter the location of a model file or folder already available in this session.")
                selected = prepare_model(model, location.value)
                first_trait = next(iter(selected["paths"]))
                model_for_trait(selected, first_trait)
                model_selection = selected
                print(f"Model ready: {model}. Choose a metabolite trait in step 4.")
            except (ValueError, RuntimeError, FileNotFoundError) as error:
                print(str(error))

    button.on_click(load)
    accordion = widgets.Accordion(children=[widgets.VBox([location, button, output])], selected_index=None)
    accordion.set_title(0, "Load local model")
    display(accordion)
    return {"location": location, "button": button}


def show_advanced_input():
    import ipywidgets as widgets
    from IPython.display import display

    location = widgets.Text(description="Input data file:", placeholder="A prepared dataset already in this session", layout=widgets.Layout(width="95%"))
    button = widgets.Button(description="Use the supplied dataset", layout=widgets.Layout(width="auto"))
    output = widgets.Output()

    def load(_):
        with output:
            output.clear_output(wait=True)
            try:
                if not location.value.strip():
                    raise ValueError("Select a prepared input data file already available in this session.")
                load_example_input(location.value)
            except (ValueError, FileNotFoundError) as error:
                print(str(error))

    button.on_click(load)
    accordion = widgets.Accordion(children=[widgets.VBox([location, button, output])], selected_index=None)
    accordion.set_title(0, "Load local data")
    display(accordion)
    return {"location": location, "button": button}


def chosen_model(source, action="prediction"):
    model_name = globals().get("model_size", plan["models"][0]["id"])
    if model_selection is None or globals().get("MODEL") != model_name or model_selection["model"] != model_name:
        raise RuntimeError(f"The model could not be loaded. Select {model_name} and run Load model again.")
    if data_selection is None or prepared is None or data_selection["path"] != prepared:
        raise RuntimeError(f"Load the example data in the Input data section, then run {action} again.")
    if source == "Pretrained weights":
        return model_selection
    if source != "Model trained in this session":
        raise ValueError("Choose Pretrained weights or Model trained in this session.")
    if trained_selection is None or trained_selection["model"] != MODEL:
        raise RuntimeError("Train the selected model in the optional training section first.")
    if trained_selection["data_revision"] != data_selection["revision"]:
        raise RuntimeError("Input data changed after training. Train for the current input data before using this model.")
    return trained_selection


def prediction_table(result):
    with Path(result["csv"]).open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        fields, rows = reader.fieldnames or [], list(reader)
    if not {"accession", "prediction"} <= set(fields) or len(fields) != len(set(fields)):
        raise ValueError("Load a predictions CSV with one accession column and one prediction column.")
    ids = np.asarray([row["accession"] for row in rows], dtype=str)
    estimates = np.asarray([float(row["prediction"]) for row in rows])
    if not len(rows) or len(set(ids)) != len(ids) or not np.isfinite(estimates).all():
        raise ValueError("Load prediction results with one unique ID and a finite prediction for each accession.")
    train_ids, test_ids = set(result["train_ids"]), set(result["test_ids"])
    roles = ["test" if name in test_ids else "train" if name in train_ids else "query" for name in ids]
    labels, sd = np.full(len(rows), np.nan), np.nan
    enriched = {"model", "trait", "role", "observed"} <= set(fields)
    if enriched:
        if any(row["model"] != result["model"] or row["trait"] != result["trait"] for row in rows):
            raise ValueError("Select prediction results for one model and one metabolite trait.")
        if [row["role"] for row in rows] != roles:
            raise ValueError("Load prediction results with the original training and test accession roles.")
        labels = np.asarray([float(row["observed"]) if row["observed"].strip() else np.nan for row in rows])
        if np.isinf(labels).any():
            raise ValueError("Measured values must be finite or missing.")
        lookup = dict(zip(ids, labels))
    else:
        with np.load(result["reference_data"], allow_pickle=False) as data:
            label_ids = np.asarray(data["accessions"], dtype=str)
            columns = np.flatnonzero(np.asarray(data["traits"], dtype=str) == result["trait"])
            if len(columns) != 1 or len(set(label_ids)) != len(label_ids):
                raise ValueError("Load input data with one measurement column for each metabolite trait.")
            lookup = dict(zip(label_ids, np.asarray(data["phenotypes"], dtype=float)[:, columns[0]]))
            labels = np.asarray([lookup.get(name, np.nan) for name in ids])
    training_labels = np.asarray([lookup.get(name, np.nan) for name in result["train_ids"]])
    if len(training_labels) > 1 and np.isfinite(training_labels).all():
        sd = float(training_labels.std(ddof=1))
    return ids, estimates, labels, roles, sd, enriched


def restore_prediction(path):
    path = Path(path).expanduser()
    if not path.is_file():
        raise FileNotFoundError("Run prediction or select saved predictions to generate the figure. Use the CSV saved with Download predictions.")
    with path.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        required = {"model", "trait", "accession", "role", "observed", "prediction"}
        if not required <= set(reader.fieldnames or []):
            raise ValueError("Use the CSV saved by Download predictions; it includes the model, trait and measured values needed to redraw the plot.")
        rows = list(reader)
    if not rows or len({row["model"] for row in rows}) != 1 or len({row["trait"] for row in rows}) != 1:
        raise ValueError("Select saved predictions for one model and one metabolite trait, then generate the figure again.")
    model, trait = rows[0]["model"], rows[0]["trait"]
    if model not in {item["id"] for item in plan["models"]} or trait not in plan["traits"]:
        raise ValueError("Select results for a supported model and metabolite trait.")
    result = {"csv": path, "model": model, "trait": trait, "source": "saved results",
              "train_ids": [plan["accessions"][i] for i in plan["fold_partitions"][0]["train"]],
              "test_ids": [plan["accessions"][i] for i in plan["fold_partitions"][0]["held"]]}
    prediction_table(result)
    return result

def show_prediction_result(result):
    from html import escape
    from IPython.display import HTML, display
    from src.score import metrics

    model, trait = result["model"], result["trait"]
    ids, predicted, observed, roles, train_sd, enriched = prediction_table(result)
    export_dir = Path("predictions")
    export_dir.mkdir(exist_ok=True)
    paired = np.asarray([role == "test" for role in roles]) & np.isfinite(observed)
    name = TRAIT_NAMES.get(trait, trait)
    heading = f"{model} | {trait} — {name}"
    export_rows, table_rows = [], []
    for accession, role, actual, estimate in zip(ids, roles, observed, predicted):
        residual = float(estimate - actual) if np.isfinite(actual) else None
        export_rows.append({"accession": accession, "prediction": float(estimate), "model": model, "trait": trait, "role": role,
                            "observed": float(actual) if np.isfinite(actual) else "",
                            "residual_prediction_minus_observed": residual if residual is not None else ""})
        values = (accession, role, f"{actual:.6g}" if np.isfinite(actual) else "NA", f"{estimate:.6g}", f"{residual:.6g}" if residual is not None else "NA")
        table_rows.append("<tr>" + "".join(f"<td style='padding:4px 10px;border-bottom:1px solid #eee'>{escape(str(x))}</td>" for x in values) + "</tr>")
    previous_csv = result.get("results_csv")
    results_csv = Path(result["csv"]) if enriched else Path(previous_csv) if previous_csv and Path(previous_csv).is_file() else unique_output(Path(result.get("output_csv", export_dir / f"{model}_{trait}_results.csv")))
    if not enriched and not (previous_csv and Path(previous_csv).is_file()):
        with results_csv.open("x", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(export_rows[0]))
            writer.writeheader()
            writer.writerows(export_rows)
    result.update(csv=results_csv, results_csv=results_csv, plot_path=None, plot_error=None,
                  n_test=int(paired.sum()), train_sd=train_sd)
    display(HTML(f"<h3>{escape(heading)}</h3><p>{len(ids)} predictions · {roles.count('train')} training · {roles.count('test')} test accessions</p>"))
    table_header = "".join(f"<th style='position:sticky;top:0;background:white;padding:6px 10px'>{x}</th>" for x in ("Accession", "Role", "Measured", "Predicted", "Residual (predicted − observed)"))
    display(HTML("<p><b>All predictions</b> — scroll to view every accession.</p><div style='max-height:340px;overflow:auto;border:1px solid #ddd'><table style='border-collapse:collapse;width:100%'><thead><tr>" + table_header + "</tr></thead><tbody>" + "".join(table_rows) + "</tbody></table></div>"))

    case_metrics = None
    if paired.sum() >= 2 and np.isfinite(train_sd) and train_sd > 0:
        if np.ptp(observed[paired]) > 0:
            case_metrics = metrics(observed[paired], predicted[paired], train_sd)
        else:
            error = predicted[paired] - observed[paired]
            case_metrics = {"pcc": None, "scc": None, "rmse": float(np.sqrt(np.mean(error ** 2)) / train_sd), "mae": float(np.mean(np.abs(error)) / train_sd)}
    pearson = case_metrics["pcc"] if case_metrics else float(np.corrcoef(observed[paired], predicted[paired])[0, 1]) if paired.sum() >= 2 and np.ptp(observed[paired]) > 0 and np.ptp(predicted[paired]) > 0 else None
    raw_rmse = float(np.sqrt(np.mean((predicted[paired] - observed[paired]) ** 2))) if paired.any() else None
    result.update(metrics=case_metrics, plot_metrics={"pearson": pearson, "rmse": raw_rmse})

    fig, plot_path = None, None
    try:
        import matplotlib.pyplot as plt
        with plt.rc_context({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False}):
            fig, (ax, stats) = plt.subplots(1, 2, figsize=(8.3, 5), width_ratios=[3, 1.2], layout="constrained")
            stats.set_axis_off()
            if paired.any():
                ax.scatter(observed[paired], predicted[paired], s=38, color="#0072B2", alpha=0.8, edgecolors="white", linewidths=0.5, label=model)
                lower = float(min(observed[paired].min(), predicted[paired].min()))
                upper = float(max(observed[paired].max(), predicted[paired].max()))
                pad = (upper - lower) * 0.06 if upper > lower else max(abs(lower) * 0.06, 1.0)
                limits = (lower - pad, upper + pad)
                ax.plot(limits, limits, color="#707070", linestyle="--", linewidth=1.1, label="1:1 reference")
                ax.set(xlim=limits, ylim=limits, xlabel="Measured value (source-table scale)", ylabel="Predicted value (source-table scale)")
                ax.set_aspect("equal", adjustable="box")
                r_text = "NA" if pearson is None else f"{pearson:.4f}"
                rmse_text = "NA" if raw_rmse is None else f"{raw_rmse:.4f}"
                stats.text(0, 0.85, f"Test accessions: n = {int(paired.sum())}\n\nPearson r = {r_text}\n\nRMSE = {rmse_text}", va="top", transform=stats.transAxes)
                stats.text(0, 0.23, "RMSE uses the same\nscale as the phenotype.", color="#555555", fontsize=9, va="top", transform=stats.transAxes)
            else:
                ax.hist(predicted, bins="auto", color="#0072B2", edgecolor="white", alpha=0.85, label=model)
                ax.set(xlabel="Predicted phenotype", ylabel="Accession count")
                stats.text(0, 0.85, "Measured test values\nare unavailable.\nShowing predictions.", va="top", transform=stats.transAxes)
            import textwrap
            ax.set_title(textwrap.fill(heading, width=58), pad=12, fontsize=11)
            ax.legend(frameon=False, loc="lower right" if paired.any() else "best")
            plot_path = unique_output(export_dir / f"{results_csv.stem}_plot.png")
            fig.savefig(plot_path, dpi=300, facecolor="white")
            result["plot_path"] = plot_path
            plt.show()
    except Exception as error:
        result["plot_error"] = f"{type(error).__name__}: {error}"
        if plot_path and plot_path.is_file() and result["plot_path"] is None:
            plot_path.unlink()
        print(f"The figure could not be displayed ({result['plot_error']}). Predictions are saved. Use Download predictions, then Retry figure, or select saved predictions in Redraw prediction plot.")
    finally:
        if fig is not None:
            plt.close(fig)

    if case_metrics is not None:
        headers = ("Test n", "Pearson r", "Spearman ρ", "RMSE / training SD", "MAE / training SD")
        values = [str(int(paired.sum()))] + ["NA" if case_metrics[key] is None else f"{case_metrics[key]:.4f}" for key in ("pcc", "scc", "rmse", "mae")]
        display(HTML("<p><b>Test-set metrics</b></p><table><tr>" + "".join(f"<th style='padding:6px 12px'>{escape(x)}</th>" for x in headers) + "</tr><tr>" + "".join(f"<td style='padding:6px 12px'>{escape(x)}</td>" for x in values) + "</tr></table>"))
    show_result_downloads(result)


def show_result_downloads(result):
    import ipywidgets as widgets
    from IPython.display import display

    items = [("Download predictions", result["results_csv"])]
    if result.get("plot_path") and Path(result["plot_path"]).is_file():
        items.append(("Download figure", result["plot_path"]))
    result["download_buttons"] = file_download_buttons(items, f"{result['model']} | {result['trait']}")
    output = widgets.Output()
    retry = widgets.Button(description="Retry figure" if result.get("plot_error") else "Redraw figure", icon="refresh", layout=widgets.Layout(width="auto"))

    def redraw(_):
        with output:
            output.clear_output(wait=True)
            try:
                saved = restore_prediction(result["results_csv"])
                show_prediction_result(saved)
                result.update(saved)
            except (ValueError, RuntimeError, OSError) as error:
                print(f"{error} Run prediction or select saved predictions to generate the figure.")

    retry.on_click(redraw)
    result["redraw_button"] = retry
    display(retry, output)


def export_trained_model(selection, trait):
    from src.train import validate_plan

    model = selection["model"]
    path = selection["paths"][trait]
    payload = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
    expected_case = {"model": model, "trait": trait, "fold": 0,
                     "training_seed": plan["fold_partitions"][0]["training_seed"]}
    if payload.get("format") != model_module.FULL_CHECKPOINT_FORMAT or payload.get("stage") != "completed_outer_refit":
        raise ValueError("Finish training before downloading the model.")
    if any(payload.get("case", {}).get(key) != value for key, value in expected_case.items()):
        raise ValueError("Select the model trained for the chosen model size and metabolite trait.")
    config, network = validate_plan(plan, model)
    if any(payload["model_config"].get(key) != config[key] for key in ("id", "width", "depth", "parameters")):
        raise ValueError("Choose the model size used to train this model.")
    if any(payload["training_config"].get(key) != network[key] for key in ("heads", "dropout")):
        raise ValueError("Select a model trained with the settings for this model size.")
    parameters = {name: payload["state_dict"][name].detach().cpu().clone()
                  for name in model_module._parameter_spec(config, network)}
    model_module._check_inference_parameters(parameters, config, network)
    export_root, number = Path("model_exports"), 2
    while (export_root / model / f"{trait}.pt").exists():
        export_root = Path(f"model_exports_run{number}")
        number += 1
    target = export_root / model / f"{trait}.pt"
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("xb") as handle:
        torch.save(parameters, handle)
    exported = torch.load(target, map_location="cpu", weights_only=True, mmap=True)
    model_module._check_inference_parameters(exported, config, network)
    if any(not torch.equal(value, exported[name]) for name, value in parameters.items()):
        raise ValueError("The model could not be saved correctly. Download the trained model again.")
    return target


def show_model_download(selection):
    import ipywidgets as widgets
    from IPython.display import display

    names = [trait for trait in plan["traits"] if trait in selection["paths"]]
    if not names:
        raise ValueError("Train a model before selecting Download model.")
    trait = widgets.Dropdown(options=names, description="Trait:")
    button = widgets.Button(description="Download model", icon="download", layout=widgets.Layout(width="auto"))
    output = widgets.Output()

    def download(_):
        name = trait.value
        with output:
            output.clear_output(wait=True)
            path = export_trained_model(selection, name)
            save_download(path)

    button.on_click(download)
    print(f"Trained model: {selection['model']}. Choose a trait to download its model.")
    display(widgets.HBox([trait, button]), output)
    return {"trait": trait, "button": button}


def show_evaluation_result(path, expected_model):
    from html import escape
    from IPython.display import HTML, display

    path = Path(path)
    saved = json.loads(path.read_text(encoding="utf-8"))
    cases, summaries = saved.get("per_trait", []), saved.get("rows", [])
    if len(summaries) != 1 or summaries[0].get("model") != expected_model:
        raise ValueError("Select evaluation results for the chosen model.")
    if len(cases) != 36 or len({row.get("trait") for row in cases}) != 36 or {row.get("trait") for row in cases} != set(plan["traits"]):
        raise ValueError("Evaluation must contain one result for each of the 36 supported metabolite traits.")
    metric_keys = ("pcc", "scc", "rmse", "mae")
    for row in cases:
        if row.get("model") != expected_model or row.get("n_test") != 65 or row.get("n_train") != 259 or row.get("fold") != 0:
            raise ValueError("Rerun evaluation with the selected model and example input data.")
        if not all(isinstance(row.get(key), (int, float)) and np.isfinite(row[key]) for key in metric_keys):
            raise ValueError("Evaluation metrics must be finite numeric values.")
    summary = summaries[0]
    if summary.get("traits") != 36 or summary.get("fold") != 0:
        raise ValueError("Run evaluation for all 36 supported metabolite traits using the example input data.")
    if not all(isinstance(summary.get("mean", {}).get(key), (int, float)) and np.isfinite(summary["mean"][key]) for key in metric_keys):
        raise ValueError("Evaluation mean metrics must be finite numeric values.")
    means = {"model": summary["model"], "traits": summary["traits"], **summary["mean"]}
    labels = ("Model", "Trait", "Test n", "Pearson r", "Spearman ρ", "RMSE / training SD", "MAE / training SD")
    table = []
    for row in cases:
        values = [row["model"], row["trait"], row["n_test"], *[f"{row[key]:.4f}" for key in metric_keys]]
        table.append("<tr>" + "".join(f"<td style='padding:4px 10px'>{escape(str(value))}</td>" for value in values) + "</tr>")
    header = "".join(f"<th style='padding:6px 10px'>{label}</th>" for label in labels)
    display(HTML(f"<h3>{escape(expected_model)}: results for 36 traits</h3><div style='max-height:340px;overflow:auto'><table><tr>{header}</tr>{''.join(table)}</table></div>"))
    mean_labels = ("Model", "Traits", *labels[3:])
    mean_values = [means["model"], means["traits"], *[f"{means[key]:.4f}" for key in metric_keys]]
    display(HTML("<h4>Mean across 36 traits</h4><table><tr>" + "".join(f"<th style='padding:6px 10px'>{label}</th>" for label in mean_labels) + "</tr><tr>" + "".join(f"<td style='padding:6px 10px'>{escape(str(value))}</td>" for value in mean_values) + "</tr></table>"))
    export_dir = Path("evaluation")
    export_dir.mkdir(exist_ok=True)
    per_trait = unique_output(export_dir / f"{expected_model}_per_trait.csv")
    mean_path = unique_output(export_dir / f"{expected_model}_mean.csv")
    fields = ("model", "trait", "n_train", "n_test", *metric_keys)
    for target, rows, names in ((per_trait, cases, fields), (mean_path, [means], tuple(means))):
        with target.open("x", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=names, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(rows)
    buttons = file_download_buttons([("Download per-trait results", per_trait), ("Download mean results", mean_path)], expected_model)
    return {"per_trait_csv": per_trait, "mean_csv": mean_path, "download_buttons": buttons}


model_selection = None
trained_selection = None
data_selection = None
prepared = None
prediction_result = None


In [ ]:
# @title ## 2. Load model { display-mode: "form" }
# @markdown Choose SVDKL-0.01M, SVDKL-4.35M, or SVDKL-46.46M. Each model size supports the same 36 metabolite traits; start with SVDKL-0.01M for the first example. This step downloads the selected pretrained weights from [Zenodo](https://zenodo.org/records/23669233), unpacks them, and checks that the model can be loaded. A successful load displays **Model ready**. Use **Load local model** to select weights already available in this session.
model_size = "SVDKL-0.01M"  # @param ["SVDKL-0.01M", "SVDKL-4.35M", "SVDKL-46.46M"]

if "prepare_model" not in globals():
    raise RuntimeError("Run Set up SVDKL, then run Load model again.")

MODEL = model_size
model_selection, prediction_result = None, None
print(f"Loading model: {MODEL}…")
try:
    selected = prepare_model(MODEL, "")
    if selected["source"] == "unavailable":
        print(f"The model download is not configured for {MODEL}. Choose another model, or use Load local model.")
    else:
        first_trait = next(iter(selected["paths"])) if selected["source"] == "local" else plan["traits"][0]
        model_for_trait(selected, first_trait)
        print(f"Model ready: {MODEL}.")
    model_selection = selected
except (ValueError, RuntimeError, OSError) as error:
    print(f"{error} Use Load local model or run Load model again.")
advanced_model = show_advanced_model(MODEL)


In [ ]:
# @title ## 3. Input data { display-mode: "form" }
# @markdown The example contains structural variant genotypes and metabolite measurements for **324 tomato accessions and 36 traits**. This step loads the prepared example input. If it has not been prepared, click **Prepare example data** to obtain the original source files and build it, or use **Load local data** for an existing input. See the [data sources and input format](https://github.com/wyqmath/SVDKL#data) for details.
if "load_example_input" not in globals():
    raise RuntimeError("Run Set up SVDKL before loading the example data in Input data.")
try:
    load_example_input()
except FileNotFoundError:
    print("Click Prepare example data, or choose Load local data.")
    prepare_data_button = show_example_data_action()
advanced_input = show_advanced_input()


In [ ]:
# @title ## 4. Run prediction { display-mode: "form" }
# @markdown Choose a metabolite from the **ID — name** list and select **Pretrained weights**. Running this step produces a prediction for each accession in the example input. The results table appears with a plot comparing measured and predicted values for the **65 test accessions**, labelled with the chosen model and metabolite. Pearson correlation and RMSE summarize the test predictions. To use a model trained here, choose **Model trained in this session**.
trait = "SlFM0030 — Cysteine"  # @param ["SlFM0030 — Cysteine", "SlFM0047 — L-Isoleucine", "SlFM0048 — Ornithine", "SlFM0069 — Salicylic acid", "SlFM0216 — Methoxy indoleacetic acid", "SlFM0222 — Chalcone", "SlFM0244 — Acetyl-serotonin", "SlFM0259 — 2'-Deoxycytidine", "SlFM0275 — L-Homocarnosine", "SlFM0336 — ABA", "SlFM0346 — Homocystine", "SlFM0386 — N-(9H-Purin-6-ylcarbamoyl)threonine", "SlFM0416 — IAA-Asp", "SlFM0421 — Catechin", "SlFM0430 — N-Sinapoylputrescine", "SlFM0438 — 2'-O-methylguanosine", "SlFM0502 — N-Feruloyltyramine", "SlFM0533 — p-Coumaric acid 4-O-hexoside", "SlFM0555 — GA7-1", "SlFM0577 — 4-Coumaroylquinic acid", "SlFM0625 — Adenosine 3'-monophosphate", "SlFM0716 — Sinapaldehyde glucoside", "SlFM0778 — S-Adenosyl-L-homocysteine", "SlFM0982 — Naringenin C-hexoside", "SlFM1384 — Quercetin O-malonylhexoside", "SlFM1470 — Cyanidin 3-rutinoside", "SlFM1542 — Petunidin 3-O-rutinoside", "SlFM1553 — N', N'', N'''-Di-p-coumaroyl, sinapoylagmatine", "SlFM1619 — N', N'', N'''-Diferuloyl, caffeoylspermidine", "SlFM1621 — N', N'', N'''-p-hydroxybenxoyl,caffeoyl, feruloylagmatine", "SlFM1804 — Dihydroxy tomatidine-O-hexosyl-O-rhamnoside", "SlFM1826 — Delphinidin 3-sophoroside-5-rhamnoside", "SlFM1952 — Ferulylpelargonidin di-O-hexosyl-O-pentoside", "SlFM1980 — Habrochaitoside A", "SlFM1984 — Lycoperoside H", "SlFM1988 — Lycoperoside C"]
prediction_model = "Pretrained weights"  # @param ["Pretrained weights", "Model trained in this session"]
TRAIT = trait.split(" — ", 1)[0]
prediction_result = None

if "plan" not in globals() or "chosen_model" not in globals():
    raise RuntimeError("Run Set up SVDKL, then Load model and Input data before running prediction again.")
if TRAIT not in plan["traits"]:
    raise ValueError("Choose a metabolite trait from the supported trait list.")
selection = chosen_model(prediction_model)
print(f"Loading {selection['model']} for {TRAIT}…")
checkpoint = model_for_trait(selection, TRAIT)
print(f"Model ready: {selection['model']} | {TRAIT}.")
fit_label = "trained" if selection["source"] == "trained" else "pretrained"
output = unique_output(Path("predictions") / f"{fit_label}_{MODEL}_{TRAIT}.csv")
with tempfile.TemporaryDirectory(prefix="svdkl_prediction_") as temporary:
    raw_output = Path(temporary) / "prediction.csv"
    subprocess.run([sys.executable, "-m", "src.predict", "--checkpoint", str(checkpoint),
                    "--data", str(prepared), "--model", MODEL, "--trait", TRAIT,
                    "--device", "cpu", "--output", str(raw_output)], check=True, stdout=subprocess.PIPE)
    prediction_result = {"csv": raw_output, "output_csv": output, "model": MODEL, "trait": TRAIT,
                         "reference_data": prepared, "source": fit_label, "checkpoint": checkpoint,
                         "input_revision": data_selection["revision"],
                         "train_ids": [plan["accessions"][i] for i in plan["fold_partitions"][0]["train"]],
                         "test_ids": [plan["accessions"][i] for i in plan["fold_partitions"][0]["held"]]}
    show_prediction_result(prediction_result)


In [ ]:
# @title ## 5. Save the results { display-mode: "form" }
# @markdown Select **Download predictions** beside the results to save the prediction table as a CSV file, or **Download figure** to save the plot as a PNG image. This step shows the download buttons again for the latest prediction. The CSV includes accession IDs, predictions, measured values, and sample roles for later inspection and plotting.
result = globals().get("prediction_result")
if result is None or "show_result_downloads" not in globals():
    raise RuntimeError("Run prediction or select saved predictions to generate the figure.")
show_result_downloads(result)


In [ ]:
# @title ## Optional: Redraw prediction plot { display-mode: "form" }
# @markdown Choose **Current predictions** to use the latest results, or **Saved predictions** and the location of a downloaded prediction CSV. Enable **REDRAW_RESULTS** to generate the plot. The saved table includes the model, metabolite, measurements, and sample roles, allowing the figure to retain the correct labels when reopened in another session.
REDRAW_RESULTS = False  # @param {type:"boolean"}
prediction_source = "Current predictions"  # @param ["Current predictions", "Saved predictions"]
prediction_csv = ""  # @param {type:"string"}

if REDRAW_RESULTS:
    if "restore_prediction" not in globals() or "show_prediction_result" not in globals():
        raise RuntimeError("Run setup, then run prediction or select saved predictions to generate the figure.")
    if prediction_source == "Current predictions":
        current = globals().get("prediction_result")
        if current is None or not current.get("results_csv"):
            raise RuntimeError("Run prediction or select saved predictions to generate the figure.")
        selected_csv = current["results_csv"]
    elif prediction_source == "Saved predictions":
        if not prediction_csv.strip():
            raise ValueError("Select your saved predictions CSV, then run Redraw prediction plot again.")
        selected_csv = prediction_csv.strip()
    else:
        raise ValueError("Choose Current predictions or Saved predictions, then run this step again.")
    restored = restore_prediction(selected_csv)
    show_prediction_result(restored)
    prediction_result = restored


In [ ]:
# @title ## Optional: Prepare the input data { display-mode: "form" }
# @markdown Enable **PREPARE_INPUT** to build the example input from the population structural variant file and integrated phenotype table linked in the [data instructions](https://github.com/wyqmath/SVDKL#data). Source files already present in data/ are reused; missing files are downloaded from their original providers. Preparation selects the 36 metabolite columns and 324 complete accessions. Then run **Input data** to load the prepared input.
PREPARE_INPUT = False  # @param {type:"boolean"}

if PREPARE_INPUT:
    prepare_example_input()


In [ ]:
# @title ## Optional: Train a model { display-mode: "form" }
# @markdown Select a GPU runtime, choose a metabolite, and enable **RUN_TRAINING**. Training uses the fixed split described on [GitHub](https://github.com/wyqmath/SVDKL#fixed-split), with validation selecting the training duration before a fresh fit on the full training partition. The maximum selection budget is 105 epochs. Enable **TRAIN_ALL_TRAITS** to train all 36 traits. When training finishes, use **Download model** to save a model.
RUN_TRAINING = False  # @param {type:"boolean"}
training_trait = "SlFM0030 — Cysteine"  # @param ["SlFM0030 — Cysteine", "SlFM0047 — L-Isoleucine", "SlFM0048 — Ornithine", "SlFM0069 — Salicylic acid", "SlFM0216 — Methoxy indoleacetic acid", "SlFM0222 — Chalcone", "SlFM0244 — Acetyl-serotonin", "SlFM0259 — 2'-Deoxycytidine", "SlFM0275 — L-Homocarnosine", "SlFM0336 — ABA", "SlFM0346 — Homocystine", "SlFM0386 — N-(9H-Purin-6-ylcarbamoyl)threonine", "SlFM0416 — IAA-Asp", "SlFM0421 — Catechin", "SlFM0430 — N-Sinapoylputrescine", "SlFM0438 — 2'-O-methylguanosine", "SlFM0502 — N-Feruloyltyramine", "SlFM0533 — p-Coumaric acid 4-O-hexoside", "SlFM0555 — GA7-1", "SlFM0577 — 4-Coumaroylquinic acid", "SlFM0625 — Adenosine 3'-monophosphate", "SlFM0716 — Sinapaldehyde glucoside", "SlFM0778 — S-Adenosyl-L-homocysteine", "SlFM0982 — Naringenin C-hexoside", "SlFM1384 — Quercetin O-malonylhexoside", "SlFM1470 — Cyanidin 3-rutinoside", "SlFM1542 — Petunidin 3-O-rutinoside", "SlFM1553 — N', N'', N'''-Di-p-coumaroyl, sinapoylagmatine", "SlFM1619 — N', N'', N'''-Diferuloyl, caffeoylspermidine", "SlFM1621 — N', N'', N'''-p-hydroxybenxoyl,caffeoyl, feruloylagmatine", "SlFM1804 — Dihydroxy tomatidine-O-hexosyl-O-rhamnoside", "SlFM1826 — Delphinidin 3-sophoroside-5-rhamnoside", "SlFM1952 — Ferulylpelargonidin di-O-hexosyl-O-pentoside", "SlFM1980 — Habrochaitoside A", "SlFM1984 — Lycoperoside H", "SlFM1988 — Lycoperoside C"]
TRAIN_ALL_TRAITS = False  # @param {type:"boolean"}
SAVE_OPTIMIZER = False  # @param {type:"boolean"}
SKIP_COMPLETED_TRAITS = False  # @param {type:"boolean"}
training_trait = training_trait.split(" — ", 1)[0]

if RUN_TRAINING:
    trained_selection = None
    if model_selection is None or MODEL != model_size or model_selection["model"] != MODEL:
        raise RuntimeError("Choose the model size and run Load model before training.")
    if data_selection is None or prepared is None or data_selection["path"] != prepared:
        raise RuntimeError("Load the example data in the Input data section, then run training again.")
    if not TRAIN_ALL_TRAITS and training_trait not in plan["traits"]:
        raise ValueError("Choose a metabolite trait from the supported trait list.")
    selected_traits = list(plan["traits"]) if TRAIN_ALL_TRAITS else [training_trait]
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    command = [sys.executable, "-m", "src.train", "--model", MODEL,
               "--trait", "all" if TRAIN_ALL_TRAITS else training_trait,
               "--fold", "0", "--device", device, "--data", str(prepared), "--output", "results"]
    if SAVE_OPTIMIZER:
        command.append("--save-optimizer")
    if SKIP_COMPLETED_TRAITS:
        command.append("--resume")
    subprocess.run(command, check=True)
    trained_selection = {"model": MODEL, "source": "trained", "root": Path("results"),
                       "data_revision": data_selection["revision"],
                       "paths": {name: Path("results") / MODEL / "fold_0" / f"{MODEL}_f0_{name}.full.pt"
                                 for name in selected_traits}}
    prediction_result = None
    print("Training complete. In step 4, choose Model trained in this session and the trained trait to run prediction.")
    trained_model_download = show_model_download(trained_selection)


In [ ]:
# @title ## Optional: Evaluate the model { display-mode: "form" }
# @markdown Load or train all 36 trait models for the chosen model size, then enable **RUN_SCORING**. Evaluation uses the fixed **65 test accessions** and reports Pearson correlation, Spearman correlation, and RMSE and MAE normalized by the training standard deviation. The notebook displays per-trait results and their mean, with buttons to download both tables.
RUN_SCORING = False  # @param {type:"boolean"}
evaluation_model = "Pretrained weights"  # @param ["Pretrained weights", "Model trained in this session"]

if RUN_SCORING:
    selection = chosen_model(evaluation_model, action="evaluation")
    for name in plan["traits"]:
        model_for_trait(selection, name)
    label = "trained" if selection["source"] == "trained" else "pretrained"
    output = unique_output(Path(f"metrics_{label}_{MODEL}.json"))
    subprocess.run([sys.executable, "-m", "src.score", "--results", str(selection["root"]),
                    "--model", MODEL, "--data", str(prepared), "--output", str(output)], check=True, stdout=subprocess.PIPE)
    evaluation_result = show_evaluation_result(output, MODEL)
